# 🏃 Teste: LocalProvider

Este notebook testa o **LocalProvider** do AutoMLChain - fine-tuning local usando GPU.

### Requisitos:
- GPU (NVIDIA com CUDA ou Mac Silicon)
- ~5GB de espaço em disco para modelo + dependências

### Tempo estimado:
- Instalação: ~10 minutos
- Download modelo: ~5 minutos
- Fine-tuning (1 epoch, 100 samples): ~15-30 minutos (depende da GPU)

## 1. Instalar Dependências

In [ ]:
# Instalar dependências do LocalProvider
!pip install torch transformers peft accelerate datasets bitsandbytes -q

# Verificar instalação
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Preparar Dataset

In [ ]:
import json
import os

# Dataset pequeno para teste rápido
test_data = [
    {"input": "I love this product, it's amazing!", "output": "positive"},
    {"input": "This is the worst thing I ever bought.", "output": "negative"},
    {"input": "Really enjoyed using this, highly recommend.", "output": "positive"},
    {"input": "Not bad, but could be better.", "output": "neutral"},
    {"input": "Terrible experience, never again.", "output": "negative"},
    {"input": "It's okay, nothing special.", "output": "neutral"},
    {"input": "Best purchase I ever made!", "output": "positive"},
    {"input": "Very disappointed with the quality.", "output": "negative"},
    {"input": "Works as expected, good value.", "output": "positive"},
    {"input": "Would not recommend to anyone.", "output": "negative"},
]

# Salvar dataset
os.makedirs("data", exist_ok=True)
with open("data/test_sentiment.jsonl", "w") as f:
    for item in test_data:
        f.write(json.dumps(item) + "\n")

print(f"Dataset salvo: {len(test_data)} samples")

## 3. Testar LocalProvider

In [ ]:
# Clone ou atualize o AutoMLChain
!git clone https://github.com/gumeeee/automlchain.git /tmp/automlchain 2>/dev/null || cd /tmp/automlchain && git pull

import sys
sys.path.insert(0, "/tmp/automlchain")

from automlchain import AutoMLPipeline

# Criar pipeline com LocalProvider
pipeline = AutoMLPipeline(provider="local")

print("LocalProvider pronto!")
print(f"Modelo padrão: {pipeline.provider.model}")

## 4. Iniciar Fine-tuning

In [ ]:
import time

print("🚀 Iniciando fine-tuning local...")
print(f"Modelo: TinyLlama/TinyLlama-1.1B")
print(f"Dataset: data/test_sentiment.jsonl")
print("-" * 50)

# Iniciar treinamento
job = pipeline.train(
    model="TinyLlama/TinyLlama-1.1B",
    training_file="data/test_sentiment.jsonl",
    hyperparameters={
        "epochs": 3,
        "batch_size": 2,
        "learning_rate": 1e-4,
        "lora_rank": 8,
        "max_seq_length": 256,
    }
)

print(f"\n✅ Job iniciado!")
print(f"   Job ID: {job.job_id}")
print(f"   Modelo: {job.model}")

## 5. Monitorar Progresso

In [ ]:
# Monitorar progresso (execute esta célula repetidamente)
print("📊 Monitorando progresso...")
print("Pressione Kernel > Interrupt para cancelar")
print()

while True:
    status = pipeline.get_status(job.job_id)
    
    print(f"[{time.strftime('%H:%M:%S')}] Status: {status.status}")
    
    if status.status == "completed":
        print("\n🎉 Treinamento concluído!")
        break
    elif status.status == "failed":
        print(f"\n❌ Falhou: {status.error}")
        break
    
    time.sleep(30)  # Verificar a cada 30 segundos

## 6. Testar Inferência

In [ ]:
# Testar o modelo fine-tuned
print("🧪 Testando inferência...")

test_prompts = [
    "I really love this product!",
    "This is awful, I hate it.",
    "It's okay, nothing special."
]

for prompt in test_prompts:
    print(f"\n📝 Input: {prompt}")
    try:
        result = pipeline.provider.infer(
            prompt=prompt,
            job_id=job.job_id,
            max_tokens=50
        )
        print(f"   Output: {result}")
    except Exception as e:
        print(f"   Error: {e}")

## 7. Verificar Logs

In [ ]:
# Ver logs do treinamento
import os

log_dir = f"./outputs/checkpoints/{job.job_id}"
if os.path.exists(log_dir):
    print(f"📁 Logs salvos em: {log_dir}")
    print()
    
    for f in os.listdir(log_dir):
        print(f"   - {f}")
else:
    print("Logs ainda não disponíveis...")

---

## 📝 Notas

1. **Primeira execução** é lenta (download do modelo + dependências)
2. **GPU é recomendada** - CPU será muito lento
3. **Modelo TinyLlama** foi escolhido por ser pequeno (~1GB)
4. Para datasets maiores, considere modelos maiores

### Próximos passos:
- Testar com dataset maior
- Experimentar hiperparâmetros
- Comparar com TogetherProvider